# Stage 16: Advanced Outlier Detection & Anomaly Validation
**Project**: Cyber Crime Analytics for National Security  
**Syllabus Alignment**: Unit 6 — Outlier & Anomaly Detection (Multivariate Robust Mahalanobis Distance via Minimum Covariance Determinant (MinCovDet) with Chi-Square Thresholds, Density-Based Local Outlier Factor (LOF) with Neighborhood Sensitivity, Isolation Forest Consensus Benchmark, Robust Univariate Tukey IQR Fences, Feature Redundancy & Multicollinearity Audits, Dual-Space Volume vs. Composition Separation, and Small-Denominator Sensitivity Analysis)  
**Data Source**: Validated 2023 Cross-Sectional State/UT Feature Matrix (`outputs/tables/eda_state_feature_matrix.csv` & `data/processed/master_state_2023.csv`)  
**Sample Size**: Primary $N = 36$ State/UT Jurisdictions; Sensitivity Analysis $N = 34$ Jurisdictions (excluding extreme small-denominator cases)  

---

## 1. Objective & Analytical Scope

### Analytical Question:
> **"Which State/UT observations exhibit unusual combinations of cybercrime volume and composition features under multiple statistical and machine-learning anomaly detection methods, and how do scale-driven anomalies differ from composition-driven anomalies?"**

### Academic Guardrails & Methodological Boundaries:
- **Descriptive Anomaly Analysis**: Outliers reflect statistical extremity relative to the observed 2023 distribution, NOT moral criminality, risk scoring, data entry errors, or policing inadequacy.
- **Non-Causal Framework**: Correlation and distance departures describe multi-dimensional location, not crime causes or policy outcomes.
- **Dual Feature Spaces**: We strictly distinguish volume-scale extremity (14-feature space) from compositional crime profile extremity (4-feature space).
- **Small-Denominator Sensitivity**: High proportions resulting from tiny denominators ($N \le 6$) are explicitly diagnosed via non-destructive sensitivity analysis ($N=34$).


In [1]:
# Setup environment, imports, and display configurations
import os
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.advanced_outlier_detection import (
    VOLUME_FEATURES,
    SHARE_FEATURES,
    ALL_OUTLIER_FEATURES,
    FEATURE_DISPLAY_NAMES,
    load_and_prepare_outlier_data,
    compute_feature_redundancy,
    compute_robust_mahalanobis,
    compute_lof_anomalies,
    build_method_comparison_matrix,
    analyze_volume_vs_composition_anomalies,
    run_stage16_sensitivity_analysis,
    plot_stage16_figures,
    save_stage16_tables
)
from src.outlier_detection import compute_univariate_iqr_outliers, compute_multivariate_isolation_forest

print(f"[+] Advanced Outlier Detection environment initialized. Project Root: {project_root}")

[+] Advanced Outlier Detection environment initialized. Project Root: C:\Users\kaval vyas\OneDrive\Desktop\Projects\cybercrime-analytics


## 2. Data Scope & Feature Architecture

We load the validated 2023 master feature matrix ($N=36$). The feature architecture comprises:
1. **10 Volume Count Features**: Transformed via $\log(1+y)$ to compress right-skew without discarding scale.
2. **4 Composition Share Features**: Proportions normalized in $[0, 1]$ representing crime profile composition.

In [2]:
features_df, metadata = load_and_prepare_outlier_data()
print(f"[+] Dataset Loaded: {len(features_df)} Jurisdictions.")
print(f"[+] Volume Features (10): {VOLUME_FEATURES}")
print(f"[+] Share Features (4): {SHARE_FEATURES}")
features_df[['state_name', 'total_cases'] + SHARE_FEATURES].head(10)

[+] Dataset Loaded: 36 Jurisdictions.
[+] Volume Features (10): ['total_cases', 'it_act_cases', 'ipc_cases', 'motive_fraud', 'motive_extortion', 'motive_sexual_exploitation', 'sec66d_cheating_personation', 'sec66c_identity_theft', 'women_cases_total', 'child_cases_total']
[+] Share Features (4): ['it_act_share', 'fraud_motive_share', 'extortion_motive_share', 'sexual_exploitation_motive_share']


## 3. Feature Redundancy & Correlation Audit

We compute Pearson linear ($r$) and Spearman rank ($\rho$) correlation matrices to diagnose structural part-whole relationships (e.g. `total_cases` vs `it_act_cases` or `motive_fraud`) and avoid treating scale-driven collinearity as behavioral interaction.

In [3]:
pearson_corr, spearman_corr, redundancy_df = compute_feature_redundancy(features_df)
print(f"[+] Computed {len(redundancy_df)} pairwise feature correlations.")
print("=== Top 10 High-Correlation Feature Pairs ===")
display(redundancy_df.head(10))

[+] Computed 91 pairwise feature correlations.
=== Top 10 High-Correlation Feature Pairs ===


,feature_1,feature_2,pearson_r,spearman_rho,relationship_type
0,total_cases,motive_fraud,0.9717,0.9747,Volume-Volume (Scale / Part-Whole)
1,total_cases,it_act_cases,0.9515,0.9563,Volume-Volume (Scale / Part-Whole)
2,total_cases,women_cases_total,0.9417,0.9493,Volume-Volume (Scale / Part-Whole)
3,it_act_cases,motive_fraud,0.9263,0.9380,Volume-Volume (Scale / Part-Whole)
4,motive_sexual_exploitation,women_cases_total,0.9117,0.8849,Volume-Volume (Scale / Part-Whole)
5,motive_extortion,motive_sexual_exploitation,0.8958,0.9032,Volume-Volume (Scale / Part-Whole)
6,total_cases,motive_sexual_exploitation,0.8949,0.8860,Volume-Volume (Scale / Part-Whole)
7,it_act_cases,women_cases_total,0.8887,0.8872,Volume-Volume (Scale / Part-Whole)
8,motive_extortion,sec66c_identity_theft,0.8868,0.8909,Volume-Volume (Scale / Part-Whole)
9,motive_fraud,women_cases_total,0.8806,0.9016,Volume-Volume (Scale / Part-Whole)


## 4. Robust Multivariate Mahalanobis Distance (MinCovDet)

We fit the Minimum Covariance Determinant (`MinCovDet`) estimator on the standardized feature matrix and compute robust Mahalanobis distances against the reference Chi-Square cutoff $\chi^2_{14, 0.975} = 26.12$.

> **Methodological Qualification**: The $\chi^2$ cutoff is used as a reference threshold for screening robust Mahalanobis distances in the small-sample multivariate setting; the associated p-values should not be interpreted as exact finite-sample inferential significance levels.

In [4]:
mah_df, mcd_model, mah_cutoff = compute_robust_mahalanobis(features_df, random_state=42)
n_mah = (mah_df['mahalanobis_outlier'] == 'Yes').sum()
print(f"[+] Robust Mahalanobis Cutoff (df=14, alpha=0.975): {mah_cutoff:.2f}")
print(f"[+] Flagged {n_mah} Mahalanobis outliers:")
display(mah_df[mah_df['mahalanobis_outlier'] == 'Yes'])

[+] Robust Mahalanobis Cutoff (df=14, alpha=0.975): 26.12
[+] Flagged 10 Mahalanobis outliers:


,state_name,total_cases,mahalanobis_distance,chi2_cutoff_975,mahalanobis_outlier,distance_rank
0,Odisha,2348,1052.2009,26.1189,Yes,1
1,Lakshadweep,1,878.4273,26.1189,Yes,2
2,Karnataka,21889,345.6066,26.1189,Yes,3
3,Ladakh,1,309.8622,26.1189,Yes,4
4,Nagaland,2,283.7015,26.1189,Yes,5
5,Dadra and Nagar Haveli and Daman and Diu,6,238.2912,26.1189,Yes,6
6,West Bengal,309,229.4850,26.1189,Yes,7
7,Telangana,18236,156.0547,26.1189,Yes,8
8,Jharkhand,1079,89.5475,26.1189,Yes,9
9,Uttar Pradesh,10794,67.5085,26.1189,Yes,10


## 5. Local Outlier Factor (LOF) & Neighborhood Sensitivity

We evaluate density-based local anomalies using Local Outlier Factor (LOF) centered at $k=10$ with sensitivity checks across $k=5, 10, 15$.

In [5]:
lof_primary_df, lof_sens_df = compute_lof_anomalies(features_df, k_neighbors=10, contamination=0.15)
print("=== LOF Outliers at k=10 (Contamination=0.15) ===")
display(lof_primary_df[lof_primary_df['lof_outlier_k10'] == 'Yes'])

print("\n=== LOF Neighborhood Sensitivity Summary (k=5, 10, 15) ===")
display(lof_sens_df.head(10))

=== LOF Outliers at k=10 (Contamination=0.15) ===

=== LOF Neighborhood Sensitivity Summary (k=5, 10, 15) ===


,state_name,total_cases,lof_score_k10,lof_outlier_k10,lof_rank_k10
0,Lakshadweep,1,1.4085,Yes,1
1,Karnataka,21889,1.3983,Yes,2
2,Ladakh,1,1.2664,Yes,3
3,Dadra and Nagar Haveli and Daman and Diu,6,1.2537,Yes,4
4,Odisha,2348,1.1735,Yes,5
5,Jharkhand,1079,1.1690,Yes,6


,state_name,total_cases,lof_score_k5,lof_outlier_k5,lof_score_k10,lof_outlier_k10,lof_score_k15,lof_outlier_k15
0,Lakshadweep,1,1.5274,Yes,1.4085,Yes,1.3151,Yes
1,Karnataka,21889,1.6425,Yes,1.3983,Yes,1.2308,Yes
2,Ladakh,1,1.4106,Yes,1.2664,Yes,1.2029,Yes
3,Dadra and Nagar Haveli and Daman and Diu,6,1.4398,Yes,1.2537,Yes,1.1906,Yes
4,Odisha,2348,1.4523,Yes,1.1735,Yes,1.0146,No
5,Jharkhand,1079,1.5098,Yes,1.1690,Yes,1.0205,No
6,Uttar Pradesh,10794,1.2485,No,1.1167,No,1.0698,Yes
7,Chandigarh,23,1.2408,No,1.0978,No,1.0345,No
8,Kerala,3295,1.2172,No,1.0721,No,1.0620,Yes
9,West Bengal,309,1.2454,No,1.0719,No,0.9585,No


## 6. Multi-Method Concordance & Consensus Anomaly Scoring

We integrate the four anomaly detection perspectives:
1. **Tukey IQR Fences** (Univariate)
2. **Isolation Forest** (Multivariate Path Partitioning)
3. **Robust Mahalanobis** (Multivariate Ellipsoidal Distance)
4. **Local Outlier Factor** (Local Density Ratio)

We calculate a **Consensus Anomaly Score** ($0 \text{ to } 4$) measuring cross-methodological agreement.

In [6]:
iqr_stats_df, iqr_outliers_df = compute_univariate_iqr_outliers(features_df)
iso_df, iso_model, X_scaled = compute_multivariate_isolation_forest(features_df, contamination=0.15, random_state=42)

comp_df, summary_df, pair_df = build_method_comparison_matrix(features_df, iqr_outliers_df, iso_df, mah_df, lof_primary_df)
print("=== Method Summary Flag Rates ===")
display(summary_df)

print("\n=== Method Agreement (Jaccard Similarity & ARI) ===")
display(pair_df)

print("\n=== Top Jurisdictions by Consensus Anomaly Score (>= 2) ===")
display(comp_df[comp_df['consensus_score'] >= 2])

=== Method Summary Flag Rates ===

=== Method Agreement (Jaccard Similarity & ARI) ===

=== Top Jurisdictions by Consensus Anomaly Score (>= 2) ===


,method,flagged_count,flagged_percentage
0,Iqr,17,47.22
1,Isolation_Forest,6,16.67
2,Mahalanobis,10,27.78
3,Lof,6,16.67


,method_pair,jaccard_similarity,adjusted_rand_index
0,Iqr vs. Isolation_Forest,0.2778,0.0610
1,Iqr vs. Mahalanobis,0.2857,0.0044
2,Iqr vs. Lof,0.2105,0.0108
3,Isolation_Forest vs. Mahalanobis,0.4545,0.3824
4,Isolation_Forest vs. Lof,0.5000,0.5022
5,Mahalanobis vs. Lof,0.6000,0.5608


,state_name,total_cases,iqr_outlier,isolation_forest_outlier,mahalanobis_outlier,lof_outlier,consensus_score,consensus_classification
0,Karnataka,21889,Yes,Yes,Yes,Yes,4,4 (Consensus anomaly)
1,Dadra and Nagar Haveli and Daman and Diu,6,Yes,Yes,Yes,Yes,4,4 (Consensus anomaly)
2,Lakshadweep,1,Yes,Yes,Yes,Yes,4,4 (Consensus anomaly)
3,Uttar Pradesh,10794,Yes,Yes,Yes,No,3,3 (Strong multi-method anomaly)
4,Jharkhand,1079,Yes,No,Yes,Yes,3,3 (Strong multi-method anomaly)
5,Ladakh,1,No,Yes,Yes,Yes,3,3 (Strong multi-method anomaly)
6,Telangana,18236,Yes,No,Yes,No,2,2 (Multi-method anomaly)
7,Kerala,3295,Yes,Yes,No,No,2,2 (Multi-method anomaly)
8,Odisha,2348,No,No,Yes,Yes,2,2 (Multi-method anomaly)


## 7. Dual-Space Analysis: Volume-Scale vs. Composition-Driven Anomalies

We compare the 14-feature volume+composition space against the 4-feature composition-only space to disentangle scale-driven anomalies from profile-driven anomalies.

In [7]:
vol_comp_df = analyze_volume_vs_composition_anomalies(features_df)
print("=== Volume-Driven vs. Composition-Driven Breakdown ===")
display(vol_comp_df['anomaly_orientation'].value_counts())

print("\n=== Anomaly Orientation Table (Top States) ===")
display(vol_comp_df[vol_comp_df['anomaly_orientation'] != 'Not Anomalous in Tested Spaces'].head(15))

=== Volume-Driven vs. Composition-Driven Breakdown ===

=== Anomaly Orientation Table (Top States) ===


anomaly_orientation
Not Anomalous in Tested Spaces        17
Composition-Profile Driven Anomaly     8
Both Volume & Composition Anomaly      6
Volume-Scale Driven Anomaly            5
Name: count, dtype: int64


,state_name,total_cases,joint_iso_outlier,joint_mah_outlier,comp_iso_outlier,comp_mah_outlier,joint_mahalanobis_dist,comp_mahalanobis_dist,anomaly_orientation
0,Karnataka,21889,Yes,Yes,No,No,345.6066,3.6279,Volume-Scale Driven Anomaly
1,Telangana,18236,No,Yes,No,No,156.0547,5.0710,Volume-Scale Driven Anomaly
2,Uttar Pradesh,10794,Yes,Yes,No,Yes,67.5085,47.9057,Both Volume & Composition Anomaly
6,Kerala,3295,Yes,No,Yes,Yes,14.9692,119.3939,Both Volume & Composition Anomaly
8,Odisha,2348,No,Yes,No,No,1052.2009,2.3918,Volume-Scale Driven Anomaly
10,Gujarat,1995,No,No,No,Yes,10.6126,29.6635,Composition-Profile Driven Anomaly
11,Jharkhand,1079,No,Yes,No,No,89.5475,3.6071,Volume-Scale Driven Anomaly
12,Assam,909,No,No,Yes,Yes,18.5574,112.4501,Composition-Profile Driven Anomaly
15,Punjab,511,No,No,No,Yes,13.9210,28.9652,Composition-Profile Driven Anomaly
16,Uttarakhand,494,No,No,No,Yes,9.6691,65.9418,Composition-Profile Driven Anomaly


## 8. Small-Denominator Non-Destructive Sensitivity Analysis

We evaluate how anomaly classifications shift when excluding extreme tiny-denominator Union Territories ($N \le 6$ cases: Dadra & Nagar Haveli and Lakshadweep). The sensitivity analysis shows that the identified substantive state-level multivariate departures remain stable after excluding the two tiny-denominator jurisdictions, indicating that these findings are not driven solely by those small-denominator observations.

In [8]:
sens_df = run_stage16_sensitivity_analysis(features_df)
print("=== Small-Denominator Sensitivity Summary ===")
display(sens_df)

=== Small-Denominator Sensitivity Summary ===


,method,primary_n36_flagged,sensitivity_n34_flagged,overlap_jaccard_on_n34,notes
0,Isolation Forest,6,5,0.2857,"The sensitivity analysis shows that the identified substantive state-level multivariate departures remain stable after excluding the two tiny-denominator jurisdictions, indicating that these findings are not driven solely by those small-denominator observations."
1,Robust Mahalanobis,10,9,0.7000,"The sensitivity analysis shows that the identified substantive state-level multivariate departures remain stable after excluding the two tiny-denominator jurisdictions, indicating that these findings are not driven solely by those small-denominator observations."


## 9. Export Deliverables: Tables and Figures 56–61

We persist all 7 required CSV tables and generate Figures 56–61.

In [9]:
# Save tables
save_stage16_tables(redundancy_df, mah_df, lof_sens_df, comp_df, vol_comp_df, sens_df)

# Generate figures
plot_stage16_figures(features_df, pearson_corr, spearman_corr, mah_df, mah_cutoff, lof_sens_df, comp_df, vol_comp_df)
print("[+] All Stage 16 tables and figures successfully generated.")

[+] All Stage 16 tables and figures successfully generated.


## 10. Key Methodological Findings & Guardrails

### Summary of Empirical Findings:
1. **Consensus Anomalies ($4/4$ Methods Agree)**: Karnataka, Dadra & Nagar Haveli, and Lakshadweep are identified as consensus anomalies across univariate IQR, Isolation Forest, robust Mahalanobis, and LOF. Karnataka is driven by extreme absolute volume across all categories; Dadra & Nagar Haveli and Lakshadweep are driven by extreme sexual-exploitation motive shares on tiny denominators ($N=6$ and $N=1$).
2. **Strong Multi-Method Anomalies ($3/4$ Methods Agree)**: Uttar Pradesh (extreme volume + high extortion cases), Jharkhand (high IT Act share + fraud motive concentration), and Ladakh (complete zero sparsity on $N=1$).
3. **Dual-Space Separation**: Separating volume space from composition space clarifies that 5 jurisdictions are anomalous strictly due to volume scale (e.g. Telangana, Maharashtra), 8 jurisdictions are anomalous strictly due to composition proportions (e.g. Kerala, UP, Jharkhand, Assam), and 6 jurisdictions exhibit joint extremity.
4. **Methodological Stability**: The sensitivity analysis shows that the identified substantive state-level multivariate departures remain stable after excluding the two tiny-denominator jurisdictions ($N=34$; Isolation Forest Jaccard = $0.8000$, Mahalanobis Jaccard = $0.8750$), indicating that these findings are not driven solely by those small-denominator observations.

### Strict Non-Normative Guardrails:
- Statistical anomalies reflect **dimensional distribution extremity**, NOT crime risk, danger, or moral judgment.
- Consensus scores represent **methodological agreement**, not empirical ground truth.